In [1]:
import re
import pandas as pd
import string
import unicodedata
import emoji
import math
import pycld2 as cld2
from bs4 import BeautifulSoup
from sklearn.metrics import accuracy_score,classification_report

In [2]:
currency_symbols = {
    "MY": "RM",
    "HK": "$",
    "AUS": "$",
    "SG": "$",
    "NZ": "$",
    "ID": "Rp",
    "TH": "฿",
    "PH": "₱",
}

currency_codes = {
    "MY": "MYR",
    "HK": "HKD",
    "AUS": "AUD",
    "SG": "SGD",
    "NZ": "NZD",
    "ID": "IDR",
    "TH": "THB",
    "PH": "PHP",
}

time_unit_candidates = {
    "Chinese": {
        "DAILY": ["每日", "每天", "日薪", "/天"],
        "WEEKLY": ["每周", "每星期", "周薪", "/周"],
        "HOURLY": ["每小时", "时薪", "按小时", "/小时"],
        "MONTHLY": ["每月", "月薪", "按月", "/月"],
        "ANNUAL": ["每年", "年薪", "按年", "/年"],
    },
    "ChineseT": {
        "DAILY": ["每日", "每天", "日薪", "/天"],
        "WEEKLY": ["每週", "每星期", "週薪", "/週"],
        "HOURLY": ["每小時", "時薪", "按小時", "/小時"],
        "MONTHLY": ["每月", "月薪", "按月", "/月"],
        "ANNUAL": ["每年", "年薪", "按年", "/年"],
    },
    "ITALIAN": {
        "DAILY": [
            "quotidiano",
            "giornaliero",
            "al giorno",
            "per giorno",
            "gg",
            "/giorno",
        ],
        "WEEKLY": [
            "settimanale",
            "alla settimana",
            "per settimana",
            "sett.",
            "/settimana",
        ],
        "HOURLY": ["orario", "all'ora", "per ora", "tariffa oraria", "h", "/h"],
        "MONTHLY": [
            "mensile",
            "al mese",
            "per mese",
            "stipendio mensile",
            "mese",
            "/mese",
        ],
        "ANNUAL": [
            "annuale",
            "all'anno",
            "per anno",
            "stipendio annuale",
            "anno",
            "/anno",
        ],
    },
    "ENGLISH": {
        "DAILY": [
            "daily",
            "per day",
            "day rate",
            "each day",
            "a day",
            "p/d",
            "/day",
            "pd",
        ],
        "WEEKLY": [
            "weekly",
            "per week",
            "week rate",
            "each week",
            "a week",
            "p/w",
            "/week",
            "pw",
        ],
        "HOURLY": [
            "hourly",
            "per hour",
            "hourly rate",
            "an hour",
            "each hour",
            "p/h",
            "p.h.",
            "/h",
            "ph",
        ],
        "MONTHLY": [
            "monthly",
            "per month",
            "monthly salary",
            "a month",
            "each month",
            "p/m",
            "/month",
            "pm",
        ],
        "ANNUAL": [
            "annual",
            "yearly",
            "per year",
            "annually",
            "annual salary",
            "a year",
            "p.a.",
            "/year",
            "py",
        ],
    },
    "VIETNAMESE": {
        "DAILY": ["hàng ngày", "mỗi ngày", "lương ngày", "/ngày"],
        "WEEKLY": ["hàng tuần", "mỗi tuần", "lương tuần", "/tuần"],
        "HOURLY": ["hàng giờ", "mỗi giờ", "lương giờ", "/giờ"],
        "MONTHLY": ["hàng tháng", "mỗi tháng", "lương tháng", "/tháng"],
        "ANNUAL": ["hàng năm", "mỗi năm", "lương năm", "/năm"],
    },
    "INDONESIAN": {
        "DAILY": ["harian", "per hari", "gaji harian", "upah harian", "/hari"],
        "WEEKLY": [
            "mingguan",
            "per minggu",
            "gaji mingguan",
            "upah mingguan",
            "/minggu",
        ],
        "HOURLY": ["per jam", "gaji per jam", "upah per jam", "/jam"],
        "MONTHLY": ["bulanan", "per bulan", "gaji bulanan", "upah bulanan", "/bulan"],
        "ANNUAL": [
            "tahunan",
            "per tahun",
            "gaji tahunan",
            "pendapatan tahunan",
            "/tahun",
        ],
    },
    "MALAY": {
        "DAILY": ["harian", "sehari", "gaji harian", "upah harian", "/hari"],
        "WEEKLY": ["mingguan", "seminggu", "gaji mingguan", "upah mingguan", "/minggu"],
        "HOURLY": ["sejam", "gaji sejam", "upah sejam", "/jam"],
        "MONTHLY": [
            "bulanan",
            "sebulan",
            "gaji bulanan",
            "pendapatan bulanan",
            "/bulan",
        ],
        "ANNUAL": [
            "tahunan",
            "setahun",
            "gaji tahunan",
            "pendapatan tahunan",
            "/tahun",
        ],
    },
}


In [3]:
# pre-compiled regex patterns
def create_precompiled_patterns(currency_symbols, currency_codes):
    compiled_patterns = {}
    num_pattern = r"((?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{1,2})?)"  # 6,000
    sep_pattern = r"\s*[-–~to]+\s*"

    for country, sym in currency_symbols.items():
        code = currency_codes[country]
        sym = re.escape(sym)  # '$'
        code = re.escape(code)
        # $50 or 50$
        single_symbol_str = rf"(?:{sym}\s*{num_pattern}|{num_pattern}\s*{sym})"
        # USD 50 or 50 USD
        single_code_str = rf"(?:{code}\s+{num_pattern}|{num_pattern}\s+{code})"
        # $50-$60
        range_symbol_str = (
            rf"{sym}\s*{num_pattern}{sep_pattern}(?:{sym}\s*)?{num_pattern}"
        )
        # USD 50-USD60
        range_code_str = (
            rf"{code}\s+{num_pattern}{sep_pattern}(?:{code}\s+)?{num_pattern}"
        )

        patterns_for_country = {
            "range_symbol": re.compile(range_symbol_str, re.IGNORECASE),
            "range_code": re.compile(range_code_str, re.IGNORECASE),
            "single_symbol": re.compile(single_symbol_str, re.IGNORECASE),
            "single_code": re.compile(single_code_str, re.IGNORECASE),
        }
        compiled_patterns[country] = patterns_for_country
    return compiled_patterns


url_pattern = re.compile(r"https?://\S+|www\.\S+")
base64_pattern = re.compile(r"\b[A-Za-z0-9+/]{35,}={0,2}\b")
spaces_pattern = re.compile(r"\s+")
compiled_patterns = create_precompiled_patterns(currency_symbols, currency_codes)

In [4]:
def preprocessing(text):
    # None or NaN
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    text = BeautifulSoup(text, "html.parser").get_text()
    # normalize Unicode
    text = unicodedata.normalize("NFKC", text)
    # clean url
    text = url_pattern.sub("", text).strip()
    # clean emoji
    text = emoji.replace_emoji(text, replace="")
    # clean base64
    text = base64_pattern.sub("", text)
    # lowcase
    text = text.lower()
    # clean spaces
    text = spaces_pattern.sub(" ", text)
    return text

In [5]:
# Extract context until meet punctuation
def extract_context(text, start, end):
    stop_chars = set(string.punctuation) - {"/"}  # for 'p/h'
    left = start
    while left > 0 and text[left - 1] not in stop_chars:
        left -= 1
    right = end
    while right < len(text) and text[right] not in stop_chars:
        right += 1
    return text[left:right]

In [6]:
def extracting_salary(ad):
    nation = ad.get("nation_short_desc", "").strip()
    if nation not in currency_symbols:
        nation = "AUS"
    curr_code = currency_codes[nation]

    # Clean and combine data
    job_details = preprocessing(ad.get("job_ad_details", ""))
    additional_text = preprocessing(ad.get("salary_additional_text", ""))
    combined_text = additional_text + " " + job_details

    patterns = compiled_patterns[nation]
    lower = None
    upper = None
    matching = None

    # Matching number in 4 situations with 4 regex
    matching = patterns["range_symbol"].search(combined_text)
    if matching:
        lower, upper = matching.group(1), matching.group(2)
    else:
        matching = patterns["range_code"].search(combined_text)
        if matching:
            lower, upper = matching.group(1), matching.group(2)
        else:
            matching = patterns["single_symbol"].search(combined_text)
            if matching:
                lower = (
                    matching.group(1)
                    if matching.group(1) is not None
                    else matching.group(2)
                )
                upper = lower
            else:
                matching = patterns["single_code"].search(combined_text)
                if matching:
                    lower = (
                        matching.group(1)
                        if matching.group(1) is not None
                        else matching.group(2)
                    )
                    upper = lower
                else:
                    return "0-0-None-None"

    # Using matching position to get context
    contex = extract_context(combined_text, matching.start(), matching.end())

    # Time unit matching with language detection
    try:
        _, _, details = cld2.detect(job_details)
        detected_lang = details[0][0]  # get most possible language
    except Exception:
        detected_lang = "ENGLISH"  # fallback

    lang_key = detected_lang if detected_lang in time_unit_candidates else "ENGLISH"
    candidates = time_unit_candidates[lang_key]
    time_unit = None
    for unit, descriptions in candidates.items():
        for description in descriptions:
            if description in contex:
                time_unit = unit
                break
        if time_unit:
            break
    if not time_unit:
        time_unit = "MONTHLY"

    # Rounding
    lower = str(int(float(lower.replace(",", ""))))
    upper = str(int(float(upper.replace(",", ""))))
    # Return format string
    return f"{lower}-{upper}-{curr_code}-{time_unit}"

In [7]:
# Read file
test_df = pd.read_csv("../../MISC/job_data_files/salary_labelled_test_set.csv", encoding="utf-8")

# Get salary
salarys = []
for idx, ad in test_df.iterrows():
    salary = extracting_salary(ad)
    salarys.append(salary)

# Calculate accuracy
true_labels = test_df["y_true"].astype(str).tolist()
accuracy = accuracy_score(true_labels, salarys)
print(f"\nAccuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(true_labels, salarys,zero_division=0))

print("\nAfter removing all 0-0-None-None data:")
# Only use valid data
valid_indices = [
    i for i, true in enumerate(true_labels) if true.strip() != "0-0-None-None"
]
y_true_valid = [true_labels[i] for i in valid_indices]
y_pred_valid = [salarys[i] for i in valid_indices]
acc_valid = accuracy_score(y_true_valid, y_pred_valid)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true_valid, y_pred_valid,zero_division=0))





Accuracy: 76.01%

Classification Report:
                           precision    recall  f1-score   support

          0-0-NZD-MONTHLY       0.00      0.00      0.00         0
            0-0-None-None       0.92      1.00      0.96       238
         10-10-SGD-HOURLY       1.00      1.00      1.00         1
         10-40-NZD-HOURLY       0.00      0.00      0.00         1
       100-100-HKD-HOURLY       1.00      0.33      0.50         3
      100-100-HKD-MONTHLY       0.00      0.00      0.00         0
       100-120-HKD-HOURLY       0.67      1.00      0.80         2
       100-250-THB-HOURLY       0.00      0.00      0.00         2
      100-250-THB-MONTHLY       0.00      0.00      0.00         0
     1000-1000-THB-HOURLY       0.00      0.00      0.00         1
    1000-1000-THB-MONTHLY       0.00      0.00      0.00         0
100000-100000-THB-MONTHLY       0.00      0.00      0.00         1
100000-150000-PHP-MONTHLY       0.00      0.00      0.00         0
 102051-125151-AUD-